Objective: Audit the existing 10 engineered features before changing or adding anything. We need to understand their formulas, check invalid values and missingness, and assess whether they carry useful information.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "fine_tuning" / "data_splits" / "train.csv").exists():
    raise FileNotFoundError("V2 training split not found. Check the project path.")

train_df = pd.read_csv(
    PROJECT_ROOT / "fine_tuning" / "data_splits" / "train.csv"
)

print("Training data:", train_df.shape)

Training data: (215257, 122)


In [3]:
def create_engineered_features(data):
    df = data.copy()

    # Convert age and employment duration from days to years
    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

    # V1 sentinel value represents an anomalous employment duration
    employed_days = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    df["EMPLOYED_YEARS"] = -employed_days / 365.25

    # Financial ratios
    df["CREDIT_TO_INCOME"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    df["ANNUITY_TO_INCOME"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    df["CREDIT_TO_ANNUITY"] = (
        df["AMT_CREDIT"] / df["AMT_ANNUITY"].replace(0, np.nan)
    )

    df["GOODS_TO_CREDIT"] = (
        df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"].replace(0, np.nan)
    )

    # Household income ratios
    df["INCOME_PER_FAMILY_MEMBER"] = (
        df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    )

    df["INCOME_PER_CHILD"] = (
        df["AMT_INCOME_TOTAL"] / df["CNT_CHILDREN"].replace(0, np.nan)
    )

    # External score aggregation
    external_scores = df[
        ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    ]

    df["EXT_SOURCE_MEAN"] = external_scores.mean(axis=1)
    df["EXT_SOURCE_MAX"] = external_scores.max(axis=1)

    return df


train_fe = create_engineered_features(train_df)

engineered_features = [
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "CREDIT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_ANNUITY",
    "GOODS_TO_CREDIT",
    "INCOME_PER_FAMILY_MEMBER",
    "INCOME_PER_CHILD",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MAX"
]

print("Engineered training shape:", train_fe.shape)

Engineered training shape: (215257, 132)


In [4]:
# Cell 3: Engineered feature diagnostics

feature_report = pd.DataFrame({
    "Missing_Count": train_fe[engineered_features].isna().sum(),
    "Missing_Percentage": (
        train_fe[engineered_features].isna().mean() * 100
    ),
    "Unique_Values": train_fe[engineered_features].nunique(),
    "Minimum": train_fe[engineered_features].min(),
    "Median": train_fe[engineered_features].median(),
    "Maximum": train_fe[engineered_features].max()
})

display(feature_report.round(3))

,Missing_Count,Missing_Percentage,Unique_Values,Minimum,Median,Maximum
AGE_YEARS,0,0.000,17377,21.008,43.107,6.899400e+01
EMPLOYED_YEARS,38756,18.005,11769,-0.000,4.515,4.814000e+01
CREDIT_TO_INCOME,0,0.000,39372,0.005,3.265,4.922700e+01
ANNUITY_TO_INCOME,8,0.004,71916,0.000,0.163,1.452000e+00
CREDIT_TO_ANNUITY,8,0.004,33434,8.037,20.000,4.530500e+01
GOODS_TO_CREDIT,187,0.087,2767,0.167,0.894,6.667000e+00
INCOME_PER_FAMILY_MEMBER,1,0.000,2362,5142.857,75000.000,3.900000e+07
INCOME_PER_CHILD,150641,69.982,959,5921.053,121500.000,1.170000e+08
EXT_SOURCE_MEAN,122,0.057,211355,0.000,0.524,8.790000e-01
EXT_SOURCE_MAX,122,0.057,95512,0.000,0.648,9.520000e-01


In [5]:
# Cell 4: Percentile-based outlier investigation

percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]

display(
    train_fe[engineered_features]
    .quantile(percentiles)
    .T
    .round(3)
)

,0.010,0.050,0.250,0.500,0.750,0.950,0.990,0.999
AGE_YEARS,22.623,25.763,33.974,43.107,53.889,63.504,66.817,68.465
EMPLOYED_YEARS,0.304,0.567,2.097,4.515,8.695,19.962,31.031,39.803
CREDIT_TO_INCOME,0.594,1.000,2.019,3.265,5.162,9.145,13.043,19.330
ANNUITY_TO_INCOME,0.039,0.062,0.114,0.163,0.229,0.355,0.485,0.727
CREDIT_TO_ANNUITY,8.726,9.524,15.648,20.000,27.100,34.542,37.794,37.915
GOODS_TO_CREDIT,0.678,0.716,0.835,0.894,1.000,1.000,1.000,1.000
INCOME_PER_FAMILY_MEMBER,18000.000,27000.000,47943.000,75000.000,112500.000,225000.000,337500.000,675000.000
INCOME_PER_CHILD,27000.000,42750.000,78750.000,121500.000,180000.000,315000.000,450000.000,900000.000
EXT_SOURCE_MEAN,0.109,0.235,0.414,0.524,0.623,0.726,0.773,0.810
EXT_SOURCE_MAX,0.131,0.297,0.540,0.648,0.725,0.813,0.868,0.915


In [6]:
# Cell 5: Default rate across feature quantiles

analysis_df = train_fe.copy()

features_to_analyze = [
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "CREDIT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_ANNUITY",
    "GOODS_TO_CREDIT",
    "INCOME_PER_FAMILY_MEMBER",
    "INCOME_PER_CHILD",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MAX"
]

for feature in features_to_analyze:
    analysis_df[f"{feature}_BIN"] = pd.qcut(
        analysis_df[feature],
        q=5,
        duplicates="drop"
    )

    print(f"\n{feature}")
    display(
        analysis_df.groupby(
            f"{feature}_BIN",
            observed=True
        )["TARGET"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "default_rate"})
        .round(4)
    )


AGE_YEARS


,count,default_rate
AGE_YEARS_BIN,,
"(21.006999999999998, 32.011]",43053,0.1133
"(32.011, 39.466]",43059,0.0931
"(39.466, 47.138]",43047,0.0788
"(47.138, 56.041]",43046,0.0669
"(56.041, 68.994]",43052,0.0514



EMPLOYED_YEARS


,count,default_rate
EMPLOYED_YEARS_BIN,,
"(-0.001, 1.684]",35308,0.1139
"(1.684, 3.422]",35318,0.1064
"(3.422, 5.914]",35285,0.0888
"(5.914, 10.078]",35305,0.0722
"(10.078, 48.14]",35285,0.0516



CREDIT_TO_INCOME


,count,default_rate
CREDIT_TO_INCOME_BIN,,
"(0.00381, 1.818]",43148,0.0735
"(1.818, 2.764]",43023,0.0864
"(2.764, 3.907]",42983,0.0882
"(3.907, 5.769]",43062,0.0824
"(5.769, 49.227]",43041,0.0730


... (10 outputs left out)


EXT_SOURCE_MEAN


,count,default_rate
EXT_SOURCE_MEAN_BIN,,
"(-0.0009885, 0.384]",43027,0.1872
"(0.384, 0.484]",43027,0.0927
"(0.484, 0.563]",43027,0.0601
"(0.563, 0.644]",43027,0.0390
"(0.644, 0.879]",43027,0.0247



EXT_SOURCE_MAX


,count,default_rate
EXT_SOURCE_MAX_BIN,,
"(-0.0009885, 0.505]",43029,0.1715
"(0.505, 0.613]",43271,0.0912
"(0.613, 0.68]",42782,0.0645
"(0.68, 0.741]",43035,0.0447
"(0.741, 0.952]",43018,0.0315


In [7]:
# Cell 6: Correlation between engineered numerical features

correlation_features = [
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "CREDIT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_ANNUITY",
    "GOODS_TO_CREDIT",
    "INCOME_PER_FAMILY_MEMBER",
    "INCOME_PER_CHILD",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

correlation_matrix = train_fe[correlation_features].corr(
    method="spearman"
)

display(correlation_matrix.round(2))

,AGE_YEARS,EMPLOYED_YEARS,CREDIT_TO_INCOME,ANNUITY_TO_INCOME,CREDIT_TO_ANNUITY,GOODS_TO_CREDIT,INCOME_PER_FAMILY_MEMBER,INCOME_PER_CHILD,EXT_SOURCE_MEAN,EXT_SOURCE_MAX,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
AGE_YEARS,1.00,0.31,0.13,0.08,0.12,-0.03,0.10,0.11,0.29,0.24,0.60,0.10,0.21
EMPLOYED_YEARS,0.31,1.00,0.07,0.02,0.08,0.01,0.02,0.03,0.19,0.17,0.23,0.11,0.14
CREDIT_TO_INCOME,0.13,0.07,1.00,0.79,0.63,-0.12,-0.23,-0.21,0.07,0.05,0.11,0.01,0.09
ANNUITY_TO_INCOME,0.08,0.02,0.79,1.00,0.08,-0.06,-0.35,-0.32,0.04,0.02,0.04,-0.04,0.10
CREDIT_TO_ANNUITY,0.12,0.08,0.63,0.08,1.00,-0.11,0.06,0.08,0.08,0.07,0.14,0.07,0.02
GOODS_TO_CREDIT,-0.03,0.01,-0.12,-0.06,-0.11,1.00,0.03,0.02,0.06,0.05,0.05,0.06,0.02
INCOME_PER_FAMILY_MEMBER,0.10,0.02,-0.23,-0.35,0.06,0.03,1.00,0.92,0.08,0.09,0.11,0.13,-0.05
INCOME_PER_CHILD,0.11,0.03,-0.21,-0.32,0.08,0.02,0.92,1.00,0.11,0.12,0.10,0.16,-0.03
EXT_SOURCE_MEAN,0.29,0.19,0.07,0.04,0.08,0.06,0.08,0.11,1.00,0.79,0.74,0.72,0.69
EXT_SOURCE_MAX,0.24,0.17,0.05,0.02,0.07,0.05,0.09,0.12,0.79,1.00,0.65,0.61,0.60
